In [ ]:
!pip install sumy

In [ ]:
!pip install textstat

In [ ]:
import requests
from bs4 import BeautifulSoup
from textblob import TextBlob
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import textstat
import plotly.graph_objects as go
import plotly.subplots as sp
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def extract_text_from_url(url):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        paragraphs = soup.find_all('p')
        text = ' '.join([para.get_text() for para in paragraphs])
        return text
    return ""

def analyze_sentiment(text):
    blob = TextBlob(text)
    sentiment_polarity = blob.sentiment.polarity
    sentiment_subjectivity = blob.sentiment.subjectivity

    if sentiment_polarity > 0:
        sentiment = "Positive"
    elif sentiment_polarity < 0:
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    positive_prob = sentiment_polarity if sentiment_polarity > 0 else 0
    negative_prob = abs(sentiment_polarity) if sentiment_polarity < 0 else 0
    neutral_prob = 1 - (positive_prob + negative_prob)

    return sentiment, positive_prob, negative_prob, neutral_prob

def summarize_text(text, num_sentences=3):
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LsaSummarizer()
    summary = summarizer(parser.document, num_sentences)
    return ' '.join([str(sentence) for sentence in summary])

def extract_keywords(text, num_keywords=5):
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([text])
    feature_names = vectorizer.get_feature_names_out()
    scores = tfidf_matrix.toarray().flatten()
    top_keywords = [feature_names[i] for i in scores.argsort()[-num_keywords:][::-1]]
    return top_keywords

def calculate_text_similarity(text1, text2):
    vectorizer = TfidfVectorizer().fit_transform([text1, text2])
    similarity_matrix = cosine_similarity(vectorizer)
    return similarity_matrix[0, 1]

def calculate_readability(text):
    return textstat.flesch_reading_ease(text)

def calculate_lexical_diversity(text):
    words = text.split()
    return len(set(words)) / len(words) if words else 0

def plot_analysis(text, positive_prob, negative_prob, neutral_prob, readability, lexical_diversity, keywords, similarity_score=None):
    # Split text into words and sentences
    words = text.split()
    sentences = text.split('.')
    word_lengths = [len(word) for word in words]
    sentence_lengths = [len(sentence.split()) for sentence in sentences if len(sentence.split()) > 0]

    # Word Frequency Analysis
    word_freq = pd.Series(words).value_counts().head(10)

    fig = sp.make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Sentiment Distribution", "Readability Score",
            "Lexical Diversity", "Keyword WordCloud",
            "Word Frequency", "Sentence Length Distribution"
        ),
        specs=[
            [{"type": "bar"}, {"type": "indicator"}],
            [{"type": "bar"}, {"type": "xy"}],
            [{"type": "bar"}, {"type": "scatter"}]
        ]
    )

    # 1️⃣ Sentiment Distribution Bar Chart
    fig.add_trace(go.Bar(
        x=["Positive", "Negative", "Neutral"],
        y=[positive_prob, negative_prob, neutral_prob],
        marker_color=["green", "red", "gray"],
        name="Sentiment Score"
    ), row=1, col=1)

    # 2️⃣ Readability Score Gauge
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=readability,
        gauge={'axis': {'range': [0, 100]}},
    ), row=1, col=2)

    # 3️⃣ Lexical Diversity Comparison
    fig.add_trace(go.Bar(
        x=["Text", "Ideal (0.5)"],
        y=[lexical_diversity, 0.5],
        marker_color=["blue", "gray"],
        name="Lexical Diversity"
    ), row=2, col=1)

    # 4️⃣ Word Frequency Chart
    fig.add_trace(go.Bar(
        x=word_freq.index,
        y=word_freq.values,
        marker_color="purple",
        name="Top Words"
    ), row=3, col=1)

    # 5️⃣ Sentence Length Distribution
    fig.add_trace(go.Histogram(
        x=sentence_lengths,
        nbinsx=15,
        marker_color="orange",
        name="Sentence Length"
    ), row=3, col=2)

    # 6️⃣ Similarity Score Plot (if available)
    if similarity_score is not None:
        fig.add_trace(go.Bar(
            x=["Text Similarity"],
            y=[similarity_score],
            marker_color="blue",
            name="Text Similarity"
        ), row=2, col=2)

    fig.update_layout(height=1000, width=1200, title_text="Text Analysis Insights", showlegend=False)
    fig.show()

    # WordCloud for Keywords using Plotly
    wordcloud = WordCloud(width=500, height=400, background_color='white').generate(' '.join(keywords))
    wordcloud_data = np.array(wordcloud.to_array())
    fig_wc = go.Figure(go.Image(z=wordcloud_data))
    fig_wc.update_layout(title="Keyword WordCloud", width=600, height=500)
    fig_wc.show()


if __name__ == "__main__":
    choice = input("Choose an option:\n1. Enter text\n2. Enter URL\nEnter choice (1 or 2): ")

    if choice == "1":
        text = input("Enter your text: ")
    elif choice == "2":
        url = input("Enter a URL: ")
        text = extract_text_from_url(url)
    else:
        print("Invalid choice.")
        exit()

    if text:
        sentiment, positive_prob, negative_prob, neutral_prob = analyze_sentiment(text)
        summary = summarize_text(text)
        keywords = extract_keywords(text)
        readability = calculate_readability(text)
        lexical_diversity = calculate_lexical_diversity(text)

        print("\nSentiment Analysis Result:", sentiment)
        print("Positive Sentiment Probability:", positive_prob)
        print("Negative Sentiment Probability:", negative_prob)
        print("Neutral Sentiment Probability:", neutral_prob)
        print("\nText Summary:\n", summary)
        print("\nExtracted Keywords:\n", ', '.join(keywords))
        print(readability)
        print("Lexical Diversity:", lexical_diversity)

        # Comparing text similarity
        compare_choice = input("\nDo you want to compare text similarity? (y/n): ")
        if compare_choice.lower() == 'y':
            second_text = input("Enter second text: ")
            similarity_score = calculate_text_similarity(text, second_text)
            print("\nText Similarity Score:", similarity_score)
            plot_analysis(text, positive_prob, negative_prob, neutral_prob, readability, lexical_diversity, keywords, similarity_score)
        else:
            plot_analysis(text, positive_prob, negative_prob, neutral_prob, readability, lexical_diversity, keywords)
    else:
        print("No valid text provided.")

